<h1>LSTM Analysis</h1>

<h2>Description</h2>
<p>Dataset can be downloaded at: </p>
<p>Features</p>

<h2>Environment Set-up</h2>

In [1]:
import os
import sys
# Setting Hadoop home directory for the JVM
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.12"
os.environ["SPARK_HOME"] = r"C:\tools\Anaconda3\Lib\site-packages\pyspark"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep +
    os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + 
    os.environ["PATH"]
    )

In [2]:
import findspark
findspark.init()
import pyspark
import pandas as pd
from pyspark.sql.functions import col, round, when, lag, avg, first, stddev
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

#jar_path = r"D:\\PostgreSQL\\postgressql\\postgresql-42.7.12.jar"
spark =( SparkSession.builder
        .master("local[*]")
        .config('spark.hadoop.home.dir', r'C:\hadoop')
        .config('spark.jars.packages','org.postgresql:postgresql:42.7.12')
        .appName("LSTM Author Classification")
        .getOrCreate()
)
print('Spark Version:', spark.version)
print('Hadoop Home:', os.environ["HADOOP_HOME"])

Spark Version: 4.2.0
Hadoop Home: C:\hadoop


<h2>Data Extraction and Dataset Creation </h2>

In [3]:
# Set dataset path
# Copy the path from your dataset folder and past here or it wont work
DATASET_PATH = r"C:\Users\lab_services_student\Desktop\PDAN02_POE_PART_01\Authorship_TextAttribution\dataset"
# Read text file
raw_df = (spark.read.format('binaryFile')
          .option('recursiveFileLookup','true')
          .option('pathGlobalFilter','*.txt')
          .load(DATASET_PATH)
          .withColumn("content", F.expr("decode(content, 'UTF-8')"))
          .select("path", "content")
)

In [4]:
# Parsing author and book_id from file path
parsed_df =(
    raw_df
    .withColumn('normal_path',F.regexp_replace('path',r'\\','/'))
    .withColumn('author',F.regexp_extract(F.col("normal_path"), r'/dataset/([^/]+)/', 1))
    .withColumn('file_name',F.regexp_extract(F.col("normal_path"), r'/([^/]+)$', 1))
    .withColumn('book_name',F.regexp_replace(F.col("file_name"), r'\.[^.]+$', ""))
    .filter(F.col('author')!="")
    .filter(F.col("content").isNotNull())
    
)

In [5]:
# Setting up Chunk Size - Earch Row will have 200 words
chunk_size = 200
# Split text into words
words_df = (
    parsed_df
    .withColumn("words",F.split(
        F.trim(F.regexp_replace(F.col("content"), r"\s+"," ")), " ")
    )
    .filter(F.size(F.col("words")) >= chunk_size)
)

In [6]:
# Create non-overlapping chunks of 200 words
chunks_df = (
    words_df
    .withColumn("starts",
            F.sequence(
                F.lit(0),
                F.size(F.col("words")) - chunk_size,
                F.lit(chunk_size)
            )
    )
    .withColumn("chunks",
                F.transform(
                    F.col("starts"),
                    lambda s: F.array_join(F.slice(F.col('words'), s + 1, chunk_size), " ")
                )
    )
    .select("author","book_name", F.explode("chunks").alias("text"))
    .filter(F.length(F.trim(F.col("text"))) > 0)
)

In [7]:
# Displaying Spark Dataframe
chunks_df.printSchema()
chunks_df.show(5, truncate=120)

root
 |-- author: string (nullable = true)
 |-- book_name: string (nullable = true)
 |-- text: string (nullable = true)

+-----------+-------------+------------------------------------------------------------------------------------------------------------------------+
|     author|    book_name|                                                                                                                    text|
+-----------+-------------+------------------------------------------------------------------------------------------------------------------------+
|Leo Tolstoy|War and Peace|An Anonymous Volunteer, and David Widger WAR AND PEACE By Leo Tolstoy/Tolstoi CONTENTS BOOK ONE: 1805 CHAPTER I CHAPT...|
|Leo Tolstoy|War and Peace|CHAPTER II CHAPTER III CHAPTER IV CHAPTER V CHAPTER VI CHAPTER VII CHAPTER VIII CHAPTER IX CHAPTER X CHAPTER XI CHAPT...|
|Leo Tolstoy|War and Peace|CHAPTER X CHAPTER XI CHAPTER XII CHAPTER XIII CHAPTER XIV CHAPTER XV CHAPTER XVI CHAPTER XVII CHAPTER XVIII

In [ ]:
# Saving Dataframe as CSV
chunks_df.write.mode("overwrite").option('header', True).csv(r"C:\Users\lab_services_student\Desktop\PDAN02_POE_PART_01\Authorship_TextAttribution\dataset\authors.csv")

<h2>Exlporatory Data Analysis</h2>

In [8]:
# Count Chunks per Author
chunks_df.groupBy('author').count().orderBy(F.desc('count')).show(truncate=False)

+-------------------+-----+
|author             |count|
+-------------------+-----+
|Leo Tolstoy        |8526 |
|Fyodor Dostoyevsky |7530 |
|Jonathan Swift     |6209 |
|Herman Melville    |6038 |
|Nathaniel Hawthorne|5358 |
|Jack London        |5329 |
|Arthur Conan Doyle |3496 |
|Bernard Shaw       |2463 |
+-------------------+-----+



In [11]:
print("Number of rows")
chunks_df.count()

Number of rows


44949